In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
root = Path('..')
data_path = root / 'data' / 'names.txt'
words = open(data_path).read().splitlines()

print(f'Words count: {len(words)}')

In [ ]:
# build vocabulary of characters and mappings to/from ints

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

In [ ]:
# build dataset

block_size = 3
X, Y = [], []

for w in words[:5]:

    print(w)
    context = [0] * block_size # padded context
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(f"{''.join(itos[i] for i in context)} ---> {itos[ix]}")
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

print(X.shape, X.dtype, Y.shape, Y.dtype)

In [ ]:
g = torch.Generator().manual_seed(42)
C = torch.randn((27, 2)) # 27 characters, 2 dimensions
print(C.shape)

W1 = torch.randn((6, 100))  # transforms flattened 6 features to 100 hidden units
b1 = torch.randn(100)       # bias term for the hidden layer
W2 = torch.randn((100, 27)) # transforms 100 hidden units to 27 output units
b2 = torch.randn(27)        # bias term for the output layer

parameters = [C, W1, b1, W2, b2]

In [ ]:
sum(p.nelement() for p in parameters) # total number of parameters

In [ ]:
emb = C[X]
print(emb.shape)

hidden_layer = torch.tanh(emb.view(-1, 6) @ W1 + b1)
print(hidden_layer.shape)

logits = hidden_layer @ W2 + b2
print(logits.shape)

counts = logits.exp()
probs = counts / counts.sum(-1, keepdim=True)
print(probs.shape)

loss = -probs[torch.arange(len(Y)), Y].log().mean()
print(loss)

